# Committor from an Underdamped Langevin MD Trajectory (1D, infretis ASE system)

Fits the committor of a particle in 1D with **underdamped Langevin** dynamics to one long plain MD trajectory (no path sampling), with NPNEq (`CommittorNE`, CuPy). At low friction the committor depends on position and velocity, q(x, v). Two fits bring the velocity in differently:
- **History, x only** (`HISTORY`): the only CV is x; each iteration draws a lag d and the basis is a polynomial in (x(t−d), x(t)) (`history_type='y(t-d),y(t)'`; d = 0: (q(t), x(t))). With one frame per step this contains the finite-difference velocity (x(t) − x(t−d))/(d dt).
- **x and v as CVs, no history:** the CVs are x and the recorded velocity; each iteration draws one of them.

**Data:** `MD_FILE`, a float32 `.npy` of shape (frames, 2): x and v of the particle, one row per frame. Write it for an infretis ASE system with `data/make_md_path.py <run folder with infretis.toml> md_path.npy` (the calculator, mass, temperature, timestep and Langevin friction of the `[engine]` section, ASE's Langevin integrator, as in the infretis ASE engine). Default: `data/infretis_inputs/well_lgv/md_path.npy`. A: x ≤ `LAMBDA_A`, B: x ≥ `LAMBDA_B`.

**Reference (optional):** a brute-force q(x, v) of the same system (`.npz` with `xs`, `vs`, `q`) in `REF_FILE`, with v in the units of the trajectory times `V_SCALE`. Without it the fit is judged by the metrics and the plots.

**Why `CommittorNE`:** time reversal flips the velocity, so the equilibrium estimator does not apply to q(x, v).


## 1. Imports

In [ ]:
import os
import sys
import time

import numpy as np
import matplotlib.pyplot as plt
import cupy as cp
from scipy.interpolate import RegularGridInterpolator

plt.rc('font', size=10)
plt.rc('axes', titlesize=12)
plt.rc('axes', labelsize=12)
plt.rc('legend', fontsize=9)

# Run from examples/: the CuPy optimalrcs package and the helpers of the examples
REPO = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
sys.path.insert(0, f'{REPO}/optimalrcs-cupy/')
sys.path.insert(0, f'{REPO}/examples/')
import optimalrcs, plots
import seed_runs
FIG = f'{REPO}/examples/figures_by_potential'


## 2. Settings

| Name | Meaning |
|---|---|
| `MD_FILE`, `N_FRAMES` | the trajectory (x, v per frame) and how many of its first frames to use |
| `LAMBDA_A`, `LAMBDA_B` | states A (x ≤ λ_A) and B (x ≥ λ_B) |
| `DT_FRAME` | time per frame, for the plots of the library (in the units of the dynamics) |
| `V_SCALE`, `REF_FILE` | v × `V_SCALE` is the velocity axis of the reference `.npz` (`xs`, `vs`, `q`); `REF_FILE = None`: no reference |
| `HISTORY` | fit A: lags d drawn each iteration (0 = no lag) |
| `MAX_ITER_HIST`, `MAX_ITER_XV` | main iterations of fit A (history, x only) and fit B (x and v, no history) |
| `WARMUP_ITER`, `WARMUP_ENVELOPE`, `ENVELOPE`, `GAMMA`, `CHUNK`, `SEEDS` | warm-up (wide envelope), main envelope, regularization, iterations between plots, random seeds (they set the CV and envelope centre drawn per iteration) |


In [ ]:
MD_FILE = f'{REPO}/examples/data/infretis_inputs/well_lgv/md_path.npy'    # (frames, 2): x, v
N_FRAMES = 5_000_000
LAMBDA_A, LAMBDA_B = -0.35, 0.35
DT_FRAME = 0.005                    # time per frame
V_SCALE = 1.0
REF_FILE = None                     # brute-force q(x, v) .npz with xs, vs, q

SEEDS = [0]
WARMUP_ITER, WARMUP_ENVELOPE = 1000, 1
HISTORY = [0, 5]                    # fit A
MAX_ITER_HIST, MAX_ITER_XV = 10000, 10000
CHUNK = 5000
ENVELOPE, GAMMA = 0.01, 0.1
NY = 6

RUN_NAME = os.path.basename(os.path.dirname(MD_FILE))
OUT = f'{FIG}/1d_cos_well/infretis_langevin_md/{RUN_NAME}'
os.makedirs(OUT, exist_ok=True)
print('figures and results in', OUT)


## 3. Data

The first `N_FRAMES` frames, one trajectory (`i_traj` constant). The velocity is used as a CV in fit B and for the (x, v) plots; the history fit never sees it.

In [ ]:
data = np.load(MD_FILE, mmap_mode='r')[:N_FRAMES]
lam, v = np.array(data[:, 0], dtype=float), np.array(data[:, 1], dtype=float)
lambda_a, lambda_b = LAMBDA_A, LAMBDA_B
boundary0, boundary1 = lam <= lambda_a, lam >= lambda_b
i_traj = np.ones(len(lam))                  # one trajectory
t_traj = np.arange(len(lam), dtype=float)   # frame index
print(f'{len(lam)} frames, v std {v.std():.3f}; frames in A {boundary0.mean():.3f}, in B {boundary1.mean():.3f}')
trans = (np.diff(boundary0.astype(int)) == 1).sum() + (np.diff(boundary1.astype(int)) == 1).sum()
print(f'entries into A or B: {trans}')
fig, ax = plt.subplots(1, 1)
plots.plot_fep(ax, cp.asarray(lam), xlabel='$x$')
plt.show()


## 4. Reference (optional)

In [ ]:
if REF_FILE is None:
    q_exact = None
else:
    ref = np.load(REF_FILE)
    f_ref = RegularGridInterpolator((ref['xs'], ref['vs']), ref['q'], bounds_error=False, fill_value=None)
    vv = v * V_SCALE
    q_exact = np.clip(f_ref(np.c_[np.clip(lam, ref['xs'][0], ref['xs'][-1]), np.clip(vv, ref['vs'][0], ref['vs'][-1])]), 0, 1)
free = (lam > lambda_a) & (lam < lambda_b)
edges_l = np.linspace(lambda_a, lambda_b, 71)
edges_v = np.linspace(*np.quantile(v[free], [0.002, 0.998]), 61)
n_lv = np.histogram2d(lam[free], v[free], [edges_l, edges_v])[0]


def make_q():
    return optimalrcs.CommittorNE(boundary0=boundary0, boundary1=boundary1, i_traj=i_traj, t_traj=t_traj)


def report(q):
    if q_exact is None:
        return ''
    r = cp.asnumpy(q.r_traj)
    return f'RMS(fit - reference) {np.sqrt(np.mean((r[free] - q_exact[free]) ** 2)):.4f}'


def qv_map(q):
    """mean q per (x, v) bin"""
    r = cp.asnumpy(q.r_traj)
    s = np.histogram2d(lam[free], v[free], [edges_l, edges_v], weights=r[free])[0]
    fig, ax = plt.subplots(figsize=(7, 5))
    im = ax.pcolormesh(edges_l, edges_v, np.where(n_lv > 20, s / np.maximum(n_lv, 1), np.nan).T, cmap='coolwarm', vmin=0, vmax=1)
    fig.colorbar(im, ax=ax, label='$q$'); ax.set(xlabel='$x$', ylabel='$v$')
    return fig


fits = {}


def run_variant(variant, cv_cols, history, max_iter):
    """Fit `variant` for every seed. CVs: columns cv_cols of (x, v), one drawn per iteration; history: lags (None: none)."""
    Y = cp.asarray(np.c_[lam, v * V_SCALE][:, cv_cols])
    kw = dict(ny=NY, print_step=min(CHUNK, 1000), save_min_delta_zq=False, lazy_basis=True)
    if history:
        kw.update(history_delta_t=list(history), history_type='y(t-d),y(t)')
    comp_y = lambda: Y[:, np.random.randint(Y.shape[1])]
    out = seed_runs.run_seeds(optimalrcs, make_q, comp_y, kw, SEEDS, max_iter, CHUNK, f'{OUT}/{variant}', ENVELOPE, GAMMA,
                              warmup_iter=WARMUP_ITER, warmup_envelope=WARMUP_ENVELOPE, warmup_gamma=GAMMA,
                              plots=('metrics', 'feps', 'obs_pred'), plot_options=dict(feps_kw=dict(delta_t_sim=DT_FRAME)),
                              extra_plot={'q_x_v': qv_map}, report=report)
    fits[variant] = out[0] if isinstance(out, tuple) else out
    return out


## 5. Fits

Both fits use the same trajectory, seeds and settings, and differ in the CVs and the history. Plots go to `<OUT>/<variant>/seed<s>/it<N>_*.png`.

### 5a. Fit A: x only, with histories

In [ ]:
run_variant('history_x', [0], HISTORY, MAX_ITER_HIST)


### 5b. Fit B: x and v as CVs, no history

In [ ]:
run_variant('x_v', [0, 1], None, MAX_ITER_XV)


## 6. Results

Per variant and seed: the metrics at the end and, with a reference, the RMS of fit − reference over the frames between A and B. Then q along x and in the (x, v) plane (mean per bin), with the reference next to it if given.

In [ ]:
rows = []
for variant, res in fits.items():
    for s, rr in res.items():
        r = rr['r']
        row = dict(variant=variant, seed=s, final=rr.get('report') or None)
        if q_exact is not None:
            row['rms_free'] = np.sqrt(np.mean((r[free] - q_exact[free]) ** 2))
        rows.append(row)
import pandas as pd
display(pd.DataFrame(rows).set_index(['variant', 'seed']))

mid = (edges_l[1:] + edges_l[:-1]) / 2
bi = np.clip(np.digitize(lam, edges_l) - 1, 0, len(mid) - 1)
nb = np.bincount(bi[free], minlength=len(mid))
mean_q = lambda r: np.bincount(bi[free], r[free], len(mid)) / np.maximum(nb, 1)
fig, axes = plt.subplots(1, 2 if q_exact is not None else 1, figsize=(13 if q_exact is not None else 7, 4), squeeze=False)
for variant, res in fits.items():
    for s, rr in res.items():
        axes[0, 0].plot(mid, mean_q(rr['r']), label=f'{variant}, seed {s}')
        if q_exact is not None:
            axes[0, 1].plot(mid, mean_q(rr['r']) - mean_q(q_exact), label=f'{variant}, seed {s}')
if q_exact is not None:
    axes[0, 0].plot(mid, mean_q(q_exact), 'k--', label='reference')
    axes[0, 1].axhline(0, color='k', lw=0.8); axes[0, 1].set(xlabel='$x$', ylabel='fit $-$ reference'); axes[0, 1].legend()
axes[0, 0].set(xlabel='$x$', ylabel='$q$ (mean per bin)'); axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)
fig.tight_layout(); fig.savefig(f'{OUT}/profiles.png', dpi=110); plt.show()

panels = [(f'{variant}, seed {s}', rr['r']) for variant, res in fits.items() for s, rr in res.items()]
if q_exact is not None:
    panels.append(('reference', q_exact))
fig, axes = plt.subplots(1, len(panels), figsize=(5.5 * len(panels), 4), squeeze=False)
for ax, (title, r) in zip(axes[0], panels):
    s_ = np.histogram2d(lam[free], v[free], [edges_l, edges_v], weights=r[free])[0]
    im = ax.pcolormesh(edges_l, edges_v, np.where(n_lv > 20, s_ / np.maximum(n_lv, 1), np.nan).T, cmap='coolwarm', vmin=0, vmax=1)
    ax.set(title=title, xlabel='$x$', ylabel='$v$'); fig.colorbar(im, ax=ax, label='$q$')
fig.tight_layout(); fig.savefig(f'{OUT}/q_x_v.png', dpi=110); plt.show()


## 7. Save

The final q per frame of every fit, as `<OUT>/<variant>/seed<s>/q_traj.npy` (written during the fit) and here together in one file.

In [ ]:
np.savez(f'{OUT}/fits.npz', lam=lam, v=v, **{f'{variant}_seed{s}': rr['r'] for variant, res in fits.items() for s, rr in res.items()})
print('saved', f'{OUT}/fits.npz')
